In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from transformers import pipeline
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# ---------- 1) Load pipelines ----------
path_aleph = "/content/drive/MyDrive/nlpPro/alephbert_nli_full_best_early_stopping"
path_hebert = "/content/drive/MyDrive/nlpPro/heBERT_nli_full_new_data_early_stopping"

pipe_aleph = pipeline("text-classification", model=path_aleph, tokenizer=path_aleph, return_all_scores=True)
pipe_hebert = pipeline("text-classification", model=path_hebert, tokenizer=path_hebert, return_all_scores=True)

# ---------- 2) One example ensemble (returns label + confidence + full scores) ----------
def ensemble_predict_with_conf(premise, hypothesis):
    input_text = f"{premise} [SEP] {hypothesis}"

    res_aleph = pipe_aleph(input_text)[0]   # list of dicts: [{'label':..., 'score':...}, ...]
    res_hebert = pipe_hebert(input_text)[0]

    # ensure same label order
    labels = [s["label"] for s in res_aleph]
    scores_aleph = np.array([s["score"] for s in res_aleph], dtype=float)
    scores_hebert = np.array([s["score"] for s in res_hebert], dtype=float)

    final_scores = (scores_aleph + scores_hebert) / 2.0
    best_idx = int(np.argmax(final_scores))

    return labels[best_idx], float(final_scores[best_idx]), dict(zip(labels, final_scores))

# ---------- 3) Load test set ----------
test_path = "/content/drive/MyDrive/nlpPro/data3/HebNLI_test.full.clean.jsonl"
ds_test = load_dataset("json", data_files={"test": test_path})["test"]

# (אם אצלך העמודות הן "translation1"/"translation2" תשתמש בהן. אחרת "sentence1"/"sentence2")
TEXT1_COL = "translation1" if "translation1" in ds_test.column_names else "sentence1"
TEXT2_COL = "translation2" if "translation2" in ds_test.column_names else "sentence2"

# label column could be "original_label" or "label"
LABEL_COL = "original_label" if "original_label" in ds_test.column_names else "label"

# If labels are strings like "entailment"/"contradiction"/"neutral" -> ok.
# If labels are ints -> map them:
label_list = ["entailment", "contradiction", "neutral"]
if isinstance(ds_test[0][LABEL_COL], (int, np.integer)):
    id2label = {i: l for i, l in enumerate(label_list)}
    y_true = [id2label[x] for x in ds_test[LABEL_COL]]
else:
    y_true = list(ds_test[LABEL_COL])

# ---------- 4) Run over test ----------
preds = []
confs = []
scores_dicts = []

for ex in ds_test:
    premise = ex[TEXT1_COL]
    hypothesis = ex[TEXT2_COL]
    pred, conf, scores = ensemble_predict_with_conf(premise, hypothesis)
    preds.append(pred)
    confs.append(conf)
    scores_dicts.append(scores)

# ---------- 5) Metrics ----------
acc = accuracy_score(y_true, preds)
macro_f1 = f1_score(y_true, preds, average="macro")
cm = confusion_matrix(y_true, preds, labels=label_list)

print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print("\nClassification report:\n", classification_report(y_true, preds, labels=label_list))
print("\nConfusion matrix (rows=true, cols=pred):\n", cm)

# ---------- 6) Save a results table (optional) ----------
df = pd.DataFrame({
    "premise": ds_test[TEXT1_COL],
    "hypothesis": ds_test[TEXT2_COL],
    "true": y_true,
    "pred": preds,
    "ensemble_conf": confs,
})

# add per-label ensemble scores as columns (optional)
for lbl in label_list:
    df[f"ens_{lbl}"] = [d.get(lbl, np.nan) for d in scores_dicts]

out_csv = "/content/drive/MyDrive/nlpPro/ensemble_test_predictions.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# ---------- 7) (Optional) Error analysis: top confident mistakes ----------
mistakes = df[df["true"] != df["pred"]].sort_values("ensemble_conf", ascending=False)
display(mistakes.head(20))


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
Device set to use cuda:0


Generating test split: 0 examples [00:00, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Accuracy: 0.8065610859728507
Macro-F1: 0.8053060145095746

Classification report:
                precision    recall  f1-score   support

   entailment       0.83      0.81      0.82       304
contradiction       0.84      0.83      0.83       307
      neutral       0.75      0.78      0.76       273

     accuracy                           0.81       884
    macro avg       0.81      0.81      0.81       884
 weighted avg       0.81      0.81      0.81       884


Confusion matrix (rows=true, cols=pred):
 [[246  16  42]
 [ 22 255  30]
 [ 28  33 212]]
Saved: /content/drive/MyDrive/nlpPro/ensemble_test_predictions.csv


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral
673,המשמעות של מופע כזה היא שהמוזיאון צריך לבחון ה...,המוזיאון לא רצה לבחון מקרוב את תפקידו בעולם הא...,neutral,contradiction,0.989857,0.002628,0.989857,0.007515
400,זה מעביר את האחריות לבצע את ההתערבות לידי יחיד...,רופאים לעתים קרובות אוהבים לבצע התערבויות כי ה...,contradiction,neutral,0.988395,0.004596,0.007009,0.988395
347,הנחיה רחבה זו נועדה לספק את דרישות הדיווח הבסי...,ההנחיות הרחבות נועדו לתת תקני דיווח בסיסיים תו...,entailment,neutral,0.988240,0.007176,0.004584,0.988240
752,"הוא שכר חדר, וגם אני שכרתי אחד.","הוא שכר חדר, אבל לא היה לי מספיק כסף בשביל לשכ...",contradiction,neutral,0.988028,0.006024,0.005947,0.988028
647,זאת מחויבות לחינוך כללי - סדרה של קורסים שמטרת...,חינוך כללי מתמקד בפיתוח כישורי חשיבה ביקורתית ...,entailment,neutral,0.987747,0.007876,0.004378,0.987747
118,ספר אלקטרוני חינם מ- ://./,הספר הוא בכריכה רכה.,contradiction,neutral,0.987564,0.005097,0.007339,0.987564
518,"מחרוזות של דגלי תפילה נמתחות מצריח הפעמון, וסב...",דת היא דבר חשוב כאן.,entailment,neutral,0.987242,0.006756,0.006002,0.987242
809,"כל הכבוד לקירסטי אלי, שמגלמת עכשיו דוגמנית לשע...","קירסטי אלי מגלמת דוגמנית לשעבר, כיום מבוגרת, ב...",entailment,neutral,0.984821,0.006722,0.008457,0.984821
209,"בסדר, דוגמה גרועה.",זו דוגמה רעה.,neutral,entailment,0.984606,0.984606,0.002393,0.013001
509,"מ-1309 עד 1377, אביניון הייתה מושב האפיפיור.","במהלך תקופה זו, כולם אישרו את העבודה שמנהיגם עשה.",contradiction,neutral,0.980189,0.004880,0.014930,0.980189


היו **171 טעויות**.

חישוב מהיר מה-confusion matrix:

נכונות על האלכסון = (246 + 255 + 212 = 713)

סה״כ דוגמאות = (884)

טעויות = (884 - 713 = 171)


In [3]:
mistake_stats = (
    df[df["true"] != df["pred"]]
      .groupby(["true", "pred"])
      .agg(count=("pred", "size"), avg_conf=("ensemble_conf", "mean"))
      .reset_index()
      .sort_values("count", ascending=False)
)

display(mistake_stats)


,true,pred,count,avg_conf
3,entailment,neutral,42,0.712557
4,neutral,contradiction,33,0.698327
1,contradiction,neutral,30,0.728906
5,neutral,entailment,28,0.749453
0,contradiction,entailment,22,0.728595
2,entailment,contradiction,16,0.689653


In [4]:
import re
import pandas as pd
import numpy as np

# ---------- 1) עוזרים: חיפוש תבניות ----------
NEG_WORDS = ["לא", "אין", "איננו", "אינה", "אינו", "בלי", "בלתי", "אף", "שום"]
CONTRAST_WORDS = ["אבל", "אולם", "אך", "רק", "למרות", "לעומת", "מצד אחד", "מצד שני"]
QUANT_WORDS = ["כל", "כולם", "תמיד", "אף פעם", "לעולם", "לעיתים", "לפעמים", "רוב", "חלק", "מעט", "הרבה"]

def contains_any(text, words):
    t = str(text)
    return any(w in t for w in words)

def extract_numbers(text):
    # מוציא מספרים כמו 3, 3.5, 1979
    return set(re.findall(r"\d+(?:\.\d+)?", str(text)))

def has_number_mismatch(prem, hyp):
    a = extract_numbers(prem)
    b = extract_numbers(hyp)
    return (len(a) > 0 or len(b) > 0) and (a != b)

def negation_pattern(text):
    return contains_any(text, NEG_WORDS)

def contrast_pattern(text):
    return contains_any(text, CONTRAST_WORDS)

def quant_pattern(text):
    return contains_any(text, QUANT_WORDS)

def jaccard_tokens(a, b):
    # מדד "דמיון מילים" פשוט – לפעמים כשיש דמיון גבוה אבל label הוא contradiction -> זה דגל
    ta = set(re.findall(r"[א-תA-Za-z]+", str(a)))
    tb = set(re.findall(r"[א-תA-Za-z]+", str(b)))
    if not ta and not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

# ---------- 2) מוסיפים דגלים לכל שורה ----------
df2 = df.copy()

df2["is_error"] = df2["true"] != df2["pred"]

df2["prem_has_neg"] = df2["premise"].apply(negation_pattern)
df2["hyp_has_neg"]  = df2["hypothesis"].apply(negation_pattern)
df2["any_neg"]      = df2["prem_has_neg"] | df2["hyp_has_neg"]

df2["prem_has_contrast"] = df2["premise"].apply(contrast_pattern)
df2["hyp_has_contrast"]  = df2["hypothesis"].apply(contrast_pattern)
df2["any_contrast"]      = df2["prem_has_contrast"] | df2["hyp_has_contrast"]

df2["prem_has_quant"] = df2["premise"].apply(quant_pattern)
df2["hyp_has_quant"]  = df2["hypothesis"].apply(quant_pattern)
df2["any_quant"]      = df2["prem_has_quant"] | df2["hyp_has_quant"]

df2["num_mismatch"] = [
    has_number_mismatch(p, h) for p, h in zip(df2["premise"], df2["hypothesis"])
]

df2["token_jaccard"] = [
    jaccard_tokens(p, h) for p, h in zip(df2["premise"], df2["hypothesis"])
]
df2["high_overlap"] = df2["token_jaccard"] >= 0.6  # אפשר לשחק עם הסף

# ---------- 3) סיכום: כמה מהטעויות כוללות כל דגל ----------
errors = df2[df2["is_error"]].copy()

flag_cols = ["any_neg", "any_contrast", "any_quant", "num_mismatch", "high_overlap"]
summary = pd.DataFrame({
    "flag": flag_cols,
    "count_in_errors": [int(errors[c].sum()) for c in flag_cols],
    "pct_of_errors": [float(errors[c].mean()) for c in flag_cols],
}).sort_values("count_in_errors", ascending=False)

display(summary)

# ---------- 4) סיכום לפי סוג טעות (true->pred) + כמה דגלים שם ----------
by_pair = (
    errors
    .groupby(["true", "pred"])
    .agg(
        count=("pred", "size"),
        avg_conf=("ensemble_conf", "mean"),
        neg_rate=("any_neg", "mean"),
        contrast_rate=("any_contrast", "mean"),
        quant_rate=("any_quant", "mean"),
        num_mismatch_rate=("num_mismatch", "mean"),
        high_overlap_rate=("high_overlap", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

display(by_pair.head(20))

# ---------- 5) לראות דוגמאות "הכי בטוחות" מכל דגל ----------
def show_top_examples(flag, n=10):
    ex = errors[errors[flag]].sort_values("ensemble_conf", ascending=False).head(n)
    cols = ["premise", "hypothesis", "true", "pred", "ensemble_conf", "ens_entailment", "ens_contradiction", "ens_neutral", flag]
    display(ex[cols])

show_top_examples("num_mismatch", n=10)
show_top_examples("any_neg", n=10)
show_top_examples("any_contrast", n=10)
show_top_examples("high_overlap", n=10)


,flag,count_in_errors,pct_of_errors
0,any_neg,72,0.421053
2,any_quant,67,0.391813
1,any_contrast,35,0.204678
3,num_mismatch,23,0.134503
4,high_overlap,1,0.005848


,true,pred,count,avg_conf,neg_rate,contrast_rate,quant_rate,num_mismatch_rate,high_overlap_rate
3,entailment,neutral,42,0.712557,0.285714,0.119048,0.476190,0.095238,0.000000
4,neutral,contradiction,33,0.698327,0.484848,0.272727,0.454545,0.151515,0.000000
1,contradiction,neutral,30,0.728906,0.666667,0.266667,0.366667,0.100000,0.000000
5,neutral,entailment,28,0.749453,0.357143,0.214286,0.464286,0.071429,0.000000
0,contradiction,entailment,22,0.728595,0.227273,0.181818,0.272727,0.272727,0.045455
2,entailment,contradiction,16,0.689653,0.562500,0.187500,0.125000,0.187500,0.000000


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,num_mismatch
509,"מ-1309 עד 1377, אביניון הייתה מושב האפיפיור.","במהלך תקופה זו, כולם אישרו את העבודה שמנהיגם עשה.",contradiction,neutral,0.980189,0.004880,0.014930,0.980189,True
710,התוצאה הייתה ארבעת חדרי רפאל ( ).,התוצאה הייתה 3 חדרי רפאל.,contradiction,entailment,0.979713,0.979713,0.011283,0.009004,True
594,"עשור של סכסוך באפגניסטן, מ-1979 עד 1989, סיפק ...",היו הרבה סכסוכים באפגניסטן מה שהופך קיצונים אי...,entailment,neutral,0.963867,0.025699,0.010434,0.963867,True
204,"הקטנה מבין חברות הטבק הגדולות, קבוצת ליגט, הסכ...",אלסקה הייתה אחת המדינות שתבעו השבת כספים שהוצא...,neutral,entailment,0.940863,0.940863,0.044836,0.014301,True
537,"חדר החרסינה מכיל קישוטים שיוצרו ב-1760 ב- , מפ...",מפעל החרסינה הזה שכפל עיצובים סיניים של קופים.,neutral,contradiction,0.925286,0.002549,0.925286,0.072165,True
294,"מורים פרטיים, שעולים עד 415 דולר לשעה, וקורסי ...",הילדים העשירים ביותר מצליחים יותר במבחן ה- בגל...,neutral,entailment,0.908628,0.908628,0.026489,0.064883,True
826,"שלוש מאות לאחר מכן, לא הרבה השתנה. כעשרים בנקי...",ניתן למצוא כאן 17 בנקים בינלאומיים.,neutral,contradiction,0.904058,0.021980,0.904058,0.073962,True
375,"עד 2001, ה- ידרוש ממקבלי המענקים לספק מידע שיא...",ה- יכריח מקבלי מענקים לנחש את המידע שמאפשר להם...,contradiction,entailment,0.898228,0.898228,0.069348,0.032424,True
784,"אדריאן וורת'י, מנכ""לית , אמרה שהסוכנות שלה תקב...",קרנות פדרליות תלויות באוכלוסייה שהן משרתות.,entailment,neutral,0.893953,0.064764,0.041284,0.893953,True
653,"מרוצים מתקיימים בין אפריל לדצמבר ב- ליד , במרח...",יש מרוצים מאפריל עד דצמבר ב- .,entailment,contradiction,0.816124,0.126754,0.816124,0.057122,True


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,any_neg
673,המשמעות של מופע כזה היא שהמוזיאון צריך לבחון ה...,המוזיאון לא רצה לבחון מקרוב את תפקידו בעולם הא...,neutral,contradiction,0.989857,0.002628,0.989857,0.007515,True
400,זה מעביר את האחריות לבצע את ההתערבות לידי יחיד...,רופאים לעתים קרובות אוהבים לבצע התערבויות כי ה...,contradiction,neutral,0.988395,0.004596,0.007009,0.988395,True
752,"הוא שכר חדר, וגם אני שכרתי אחד.","הוא שכר חדר, אבל לא היה לי מספיק כסף בשביל לשכ...",contradiction,neutral,0.988028,0.006024,0.005947,0.988028,True
809,"כל הכבוד לקירסטי אלי, שמגלמת עכשיו דוגמנית לשע...","קירסטי אלי מגלמת דוגמנית לשעבר, כיום מבוגרת, ב...",entailment,neutral,0.984821,0.006722,0.008457,0.984821,True
523,"בעבר היו ניסיונות רבים לנקז את הביצות, אבל זה ...",ניקוז הביצות הוא דבר רע כי הוא פוגע במערכות הא...,entailment,neutral,0.976929,0.010654,0.012417,0.976929,True
779,"נשים הן חלק כה גדול מכוח העבודה, עד שקשה להאמי...",גברים הם חלק עצום מכוח העבודה ולכן הם היחידים ...,contradiction,neutral,0.973999,0.009688,0.016313,0.973999,True
164,אבל הם לא יכולים להודות בזה.,הם רוצים להיות שקופים לגבי אחריותם בנוגע לעובד...,contradiction,neutral,0.972112,0.005929,0.021959,0.972112,True
391,אנחנו עדיין צריכים למצוא את הבדיקה המדויקת ביו...,הבדיקה המדויקת ביותר לשימוש ב- עדיין לא נמצאה.,entailment,contradiction,0.971661,0.019356,0.971661,0.008982,True
745,"מי שלא יהיה הרוצח הזה, נראה שהיא הכניסה אותו ה...",הרוצח שבר חלון כדי להיכנס לבית.,contradiction,neutral,0.964263,0.005117,0.030620,0.964263,True
594,"עשור של סכסוך באפגניסטן, מ-1979 עד 1989, סיפק ...",היו הרבה סכסוכים באפגניסטן מה שהופך קיצונים אי...,entailment,neutral,0.963867,0.025699,0.010434,0.963867,True


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,any_contrast
752,"הוא שכר חדר, וגם אני שכרתי אחד.","הוא שכר חדר, אבל לא היה לי מספיק כסף בשביל לשכ...",contradiction,neutral,0.988028,0.006024,0.005947,0.988028,True
523,"בעבר היו ניסיונות רבים לנקז את הביצות, אבל זה ...",ניקוז הביצות הוא דבר רע כי הוא פוגע במערכות הא...,entailment,neutral,0.976929,0.010654,0.012417,0.976929,True
164,אבל הם לא יכולים להודות בזה.,הם רוצים להיות שקופים לגבי אחריותם בנוגע לעובד...,contradiction,neutral,0.972112,0.005929,0.021959,0.972112,True
794,"בהתחלה זה הביא להקמת ממלכה עצמאית של מיורקה, ת...",ז'אומה השני ירש את ז'אומה השלישי כמנהיג מיורקה.,contradiction,entailment,0.953639,0.953639,0.034282,0.012079,True
791,"אבל מה עם העורכים האחרים של הדיילי קאל, שלא יכ...",כמה עורכים של הדיילי קאל ציינו שהקמפוס היה פעם...,neutral,entailment,0.953339,0.953339,0.020463,0.026198,True
187,זה היה נותן לשני הצדדים הזדמנות ללמוד זה על זה.,מאמצים לגשר על הפער רק יובילו לעוינות רבה יותר...,contradiction,neutral,0.952084,0.005317,0.042599,0.952084,True
785,"המופע במיטבו בימי סוף השבוע אחר הצהריים, ואפשר...","המופע עדיין די טוב בבוקר, אבל לא במיטבו.",neutral,contradiction,0.951279,0.011769,0.951279,0.036951,True
775,"כשהוא מביט לאחור על מה שהיה לפני שנה, רובין אמ...",רובין לא רוצה לעבוד על התוכנית בעוד שנה.,neutral,contradiction,0.946210,0.010356,0.946210,0.043434,True
568,האדם הראשון שנוגע בקרקע עם חלק בגוף מלבד הרגלי...,האדם הראשון שנוגע עם הידיים בקרקע מפסיד.,neutral,entailment,0.929692,0.929692,0.053925,0.016382,True
630,"בקיץ האורז יוצר שמיכה קטיפתית ירוקה, ואז מזהיב...","בקיץ האורז זהוב וניתן לקצור אותו, אבל הוא הופך...",contradiction,entailment,0.920587,0.920587,0.022959,0.056454,True


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,high_overlap
710,התוצאה הייתה ארבעת חדרי רפאל ( ).,התוצאה הייתה 3 חדרי רפאל.,contradiction,entailment,0.979713,0.979713,0.011283,0.009004,True


In [6]:
import re

# ---------- 1) רשימות מילות זמן / "רק" / מודאליות ----------
TIME_WORDS = [
    "היום", "אתמול", "מחר", "כרגע", "עכשיו", "בעבר", "בעתיד", "לעיתים", "לפעמים", "תמיד", "אף פעם",
    "שנה", "שנים", "חודש", "חודשים", "שבוע", "שבועות", "יום", "ימים", "דקה", "דקות", "שעה", "שעות",
    "בוקר", "צהריים", "ערב", "לילה",
    "לפני", "אחרי", "במהלך", "מאז", "עד", "תוך", "בזמן", "בעת",
]
ONLY_WORDS = ["רק", "בלבד", "אך ורק", "אך", "רק עכשיו", "רק היום"]  # "אך" גם ניגוד לפעמים, אבל פה זה “בלעדיות”
MODAL_WORDS = ["יכול", "יכולה", "יכולים", "עשוי", "עשויה", "עשויים", "מותר", "אסור", "חייב", "חייבת", "חובה", "אפשר", "אי אפשר"]

def contains_phrase(text, phrase):
    return phrase in str(text)

def contains_any_phrases(text, phrases):
    t = str(text)
    return any(p in t for p in phrases)

def has_year(text):
    # שנים נפוצות: 4 ספרות 1000-2099
    return re.search(r"\b(1[0-9]{3}|20[0-9]{2})\b", str(text)) is not None

def time_mismatch(prem, hyp):
    # אם יש שנה רק באחד מהם, או שנים שונות
    prem_years = set(re.findall(r"\b(1[0-9]{3}|20[0-9]{2})\b", str(prem)))
    hyp_years  = set(re.findall(r"\b(1[0-9]{3}|20[0-9]{2})\b", str(hyp)))
    if prem_years or hyp_years:
        return prem_years != hyp_years
    # אחרת: זמן מילולי רק באחד מהם
    prem_time = contains_any_phrases(prem, TIME_WORDS)
    hyp_time  = contains_any_phrases(hyp, TIME_WORDS)
    return prem_time != hyp_time

# ---------- 2) מוסיפים דגלים ----------
df2["prem_has_time"] = df2["premise"].apply(lambda x: contains_any_phrases(x, TIME_WORDS) or has_year(x))
df2["hyp_has_time"]  = df2["hypothesis"].apply(lambda x: contains_any_phrases(x, TIME_WORDS) or has_year(x))
df2["any_time"]      = df2["prem_has_time"] | df2["hyp_has_time"]
df2["time_mismatch"] = [time_mismatch(p, h) for p, h in zip(df2["premise"], df2["hypothesis"])]

df2["prem_has_only"] = df2["premise"].apply(lambda x: contains_any_phrases(x, ONLY_WORDS))
df2["hyp_has_only"]  = df2["hypothesis"].apply(lambda x: contains_any_phrases(x, ONLY_WORDS))
df2["any_only"]      = df2["prem_has_only"] | df2["hyp_has_only"]
df2["only_mismatch"] = (df2["prem_has_only"] != df2["hyp_has_only"])  # “רק” מופיע רק בצד אחד

df2["prem_has_modal"] = df2["premise"].apply(lambda x: contains_any_phrases(x, MODAL_WORDS))
df2["hyp_has_modal"]  = df2["hypothesis"].apply(lambda x: contains_any_phrases(x, MODAL_WORDS))
df2["any_modal"]      = df2["prem_has_modal"] | df2["hyp_has_modal"]
df2["modal_mismatch"] = (df2["prem_has_modal"] != df2["hyp_has_modal"])

# מחדש את errors אחרי שהוספת עמודות
errors = df2[df2["is_error"]].copy()

new_flags = ["any_time", "time_mismatch", "any_only", "only_mismatch", "any_modal", "modal_mismatch"]

summary2 = pd.DataFrame({
    "flag": new_flags,
    "count_in_errors": [int(errors[c].sum()) for c in new_flags],
    "pct_of_errors": [float(errors[c].mean()) for c in new_flags],
}).sort_values("count_in_errors", ascending=False)

display(summary2)

def show_top_examples(flag, n=10):
    ex = errors[errors[flag]].sort_values("ensemble_conf", ascending=False).head(n)
    cols = ["premise", "hypothesis", "true", "pred", "ensemble_conf",
            "ens_entailment", "ens_contradiction", "ens_neutral", flag]
    display(ex[cols])

show_top_examples("only_mismatch", 10)
show_top_examples("time_mismatch", 10)
show_top_examples("modal_mismatch", 10)




,flag,count_in_errors,pct_of_errors
0,any_time,76,0.444444
1,time_mismatch,57,0.333333
4,any_modal,22,0.128655
5,modal_mismatch,20,0.116959
2,any_only,19,0.111111
3,only_mismatch,14,0.081871


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,only_mismatch
523,"בעבר היו ניסיונות רבים לנקז את הביצות, אבל זה ...",ניקוז הביצות הוא דבר רע כי הוא פוגע במערכות הא...,entailment,neutral,0.976929,0.010654,0.012417,0.976929,True
187,זה היה נותן לשני הצדדים הזדמנות ללמוד זה על זה.,מאמצים לגשר על הפער רק יובילו לעוינות רבה יותר...,contradiction,neutral,0.952084,0.005317,0.042599,0.952084,True
775,"כשהוא מביט לאחור על מה שהיה לפני שנה, רובין אמ...",רובין לא רוצה לעבוד על התוכנית בעוד שנה.,neutral,contradiction,0.946210,0.010356,0.946210,0.043434,True
182,"בנוסף, היא לא מציעה תוכנית נזילות לטווח ארוך ע...","התכנית אינה אחראית מבחינה פיננסית, והיא רק דוח...",entailment,neutral,0.919001,0.036352,0.044647,0.919001,True
294,"מורים פרטיים, שעולים עד 415 דולר לשעה, וקורסי ...",הילדים העשירים ביותר מצליחים יותר במבחן ה- בגל...,neutral,entailment,0.908628,0.908628,0.026489,0.064883,True
480,"בטירה יש מוזיאון מעניין, אך הוא מודרני למרבה ה...",המוזיאון שבטירה מאפשר לאנשים לרכוש שריון.,neutral,entailment,0.875003,0.875003,0.028219,0.096778,True
158,דמוקרט אסתטי אומר שאנשים רבים יותר יכולים להפי...,חווית האמנות כוללת יותר מאשר רק להסתכל עליה.,entailment,neutral,0.783509,0.154490,0.062001,0.783509,True
689,באופן כללי המשתתפים הסכימו ששיפורים בניהול תאג...,המשתתפים הסכימו רק על כמה נושאים.,neutral,contradiction,0.767808,0.019985,0.767808,0.212207,True
169,"מאחר שהסרט מסתיים במותו של שולץ, הוא נותן רק מ...",הסרט מסתיים כאשר שולץ נרצח.,neutral,entailment,0.751216,0.751216,0.184923,0.063861,True
557,"ממורנט פוינט, הכביש פונה מערבה חזרה לכיוון קינ...","קינגסטון הוא אי, כך שאפשר להגיע אליו רק בסירה.",contradiction,neutral,0.748805,0.003151,0.248044,0.748805,True


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,time_mismatch
673,המשמעות של מופע כזה היא שהמוזיאון צריך לבחון ה...,המוזיאון לא רצה לבחון מקרוב את תפקידו בעולם הא...,neutral,contradiction,0.989857,0.002628,0.989857,0.007515,True
518,"מחרוזות של דגלי תפילה נמתחות מצריח הפעמון, וסב...",דת היא דבר חשוב כאן.,entailment,neutral,0.987242,0.006756,0.006002,0.987242,True
509,"מ-1309 עד 1377, אביניון הייתה מושב האפיפיור.","במהלך תקופה זו, כולם אישרו את העבודה שמנהיגם עשה.",contradiction,neutral,0.980189,0.004880,0.014930,0.980189,True
710,התוצאה הייתה ארבעת חדרי רפאל ( ).,התוצאה הייתה 3 חדרי רפאל.,contradiction,entailment,0.979713,0.979713,0.011283,0.009004,True
523,"בעבר היו ניסיונות רבים לנקז את הביצות, אבל זה ...",ניקוז הביצות הוא דבר רע כי הוא פוגע במערכות הא...,entailment,neutral,0.976929,0.010654,0.012417,0.976929,True
196,מחברי התגובה מציינים בעיות נוספות במאמר המקורי.,ישנן עדיין בעיות נוספות במאמר המקורי לפי העורכים.,contradiction,entailment,0.975454,0.975454,0.010235,0.014311,True
779,"נשים הן חלק כה גדול מכוח העבודה, עד שקשה להאמי...",גברים הם חלק עצום מכוח העבודה ולכן הם היחידים ...,contradiction,neutral,0.973999,0.009688,0.016313,0.973999,True
164,אבל הם לא יכולים להודות בזה.,הם רוצים להיות שקופים לגבי אחריותם בנוגע לעובד...,contradiction,neutral,0.972112,0.005929,0.021959,0.972112,True
594,"עשור של סכסוך באפגניסטן, מ-1979 עד 1989, סיפק ...",היו הרבה סכסוכים באפגניסטן מה שהופך קיצונים אי...,entailment,neutral,0.963867,0.025699,0.010434,0.963867,True
794,"בהתחלה זה הביא להקמת ממלכה עצמאית של מיורקה, ת...",ז'אומה השני ירש את ז'אומה השלישי כמנהיג מיורקה.,contradiction,entailment,0.953639,0.953639,0.034282,0.012079,True


,premise,hypothesis,true,pred,ensemble_conf,ens_entailment,ens_contradiction,ens_neutral,modal_mismatch
779,"נשים הן חלק כה גדול מכוח העבודה, עד שקשה להאמי...",גברים הם חלק עצום מכוח העבודה ולכן הם היחידים ...,contradiction,neutral,0.973999,0.009688,0.016313,0.973999,True
164,אבל הם לא יכולים להודות בזה.,הם רוצים להיות שקופים לגבי אחריותם בנוגע לעובד...,contradiction,neutral,0.972112,0.005929,0.021959,0.972112,True
785,"המופע במיטבו בימי סוף השבוע אחר הצהריים, ואפשר...","המופע עדיין די טוב בבוקר, אבל לא במיטבו.",neutral,contradiction,0.951279,0.011769,0.951279,0.036951,True
808,"נו טוב, אם לא נועדנו להתחתן בגלגול הזה, אולי ה...","אם נתחתן, אקנה לך בירה או ד""ר פפר.",contradiction,entailment,0.916986,0.916986,0.022269,0.060745,True
480,"בטירה יש מוזיאון מעניין, אך הוא מודרני למרבה ה...",המוזיאון שבטירה מאפשר לאנשים לרכוש שריון.,neutral,entailment,0.875003,0.875003,0.028219,0.096778,True
168,האפשרות האיינשטיינית שהמילניום השלישי יגיע מתי...,הייתה להם תיאוריה אבל היא התבררה כלא נכונה.,contradiction,neutral,0.854704,0.057469,0.087827,0.854704,True
158,דמוקרט אסתטי אומר שאנשים רבים יותר יכולים להפי...,חווית האמנות כוללת יותר מאשר רק להסתכל עליה.,entailment,neutral,0.783509,0.154490,0.062001,0.783509,True
445,צמחי תבלין ופרחים שגדלים על הגבעות מעניקים לו ...,אתה יכול לרכוש אותו עם או בלי אגוזים בתוכו.,entailment,contradiction,0.758428,0.211816,0.758428,0.029756,True
557,"ממורנט פוינט, הכביש פונה מערבה חזרה לכיוון קינ...","קינגסטון הוא אי, כך שאפשר להגיע אליו רק בסירה.",contradiction,neutral,0.748805,0.003151,0.248044,0.748805,True
364,"כאשר אזרחים עניים נקלעים לצרות פליליות, עומדים...",אזרחים חייבים לשכור עורכי דין פרטיים אם הם רוצ...,contradiction,neutral,0.666488,0.128152,0.205361,0.666488,True
